In [ ]:
# Dependencies added for this notebook:
#   torchsynth==1.0.2            (modular synth components for §3.1)
#   pytorch-lightning==1.9.5     (torchsynth API compat, downgrade from 2.x)
#   setuptools<70                (restores pkg_resources used by torchsynth)
#
# Install into the conda env:
#   pip install torchsynth "setuptools<70" "pytorch-lightning<2.0"

import sys, os, time, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
import ipywidgets as widgets

from src.ddsp import DDSPAutoencoder, MultiScaleSpectralLoss, InharmonicityLoss
from src.training_divergence import (
    AnimalSynth, fit_synth_to_target, load_esc50_animals,
    load_audio_chunks, make_probe_trajectory, make_probe_from_clip,
    finetune_step, save_divergent_checkpoint,
)
from src.training_divergence.finetune_utils import save_inharmonic_checkpoint
from src.ddsp.dataset import URMPViolinDataset
from torch.utils.data import DataLoader

SAMPLE_RATE = 16000
HOP_LENGTH  = 64
MODELS_DIR  = "../models"
BASELINE_CKPT = os.path.join(MODELS_DIR, "ddsp_baseline_violin.pt")
PROCESSED_DIR = "../samples/processed"


def load_baseline(freeze=True):
    ckpt = torch.load(BASELINE_CKPT, map_location='cpu', weights_only=False)
    model = DDSPAutoencoder(**ckpt['config'])
    model.load_state_dict(ckpt['model_state_dict'])
    if freeze:
        model.eval()
        for p in model.parameters():
            p.requires_grad_(False)
    return model, ckpt['config']


def specshow(audio, sr=16000, title='', ax=None, fmax=8000.0):
    import librosa, librosa.display
    wav = audio.squeeze().detach().cpu().numpy()
    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=(8, 2.5))
    D = librosa.amplitude_to_db(
        np.abs(librosa.stft(wav, n_fft=1024, hop_length=256)), ref=np.max)
    librosa.display.specshow(D, sr=sr, hop_length=256,
                              x_axis='time', y_axis='log', ax=ax, fmax=fmax)
    ax.set_title(title)
    if own:
        plt.tight_layout()
        plt.show()


def play(audio, sr=16000, normalize=True):
    wav = audio.squeeze().detach().cpu().numpy()
    if normalize and np.abs(wav).max() > 1e-8:
        wav = wav / np.abs(wav).max() * 0.9
    display(Audio(wav, rate=sr))


print("Setup complete.  Baseline checkpoint:", os.path.exists(BASELINE_CKPT))


# Notebook 3: Training-Time Active Divergence

*Class: Active Divergence with Generative Deep Learning*

---

In Notebook 2 we modified a **frozen** model at inference time — bending its outputs without
changing its weights. This notebook goes deeper: we **change training itself** to deliberately
introduce divergence.

Broad et al. identify three major training-time mechanisms:

| Mechanism | Description |
|-----------|-------------|
| **Inspiring set** | Target distribution chosen to be *outside* what the synthesizer can faithfully reproduce — approximation creates novelty |
| **Domain drift / fine-tuning** | Continue training on foreign audio; model drifts toward a hybrid timbre |
| **Loss modification** | Rewarding properties the original training signal actively discourages |

All three produce **irreversible** changes to the model's internal world — new weights, not just
new inputs.

### Three techniques
1. **§3.1 Inspiring Set** — Hagiwara et al.: optimise a limited modular synth against animal vocalizations
2. **§3.2 Divergent Fine-tuning** — Fine-tune the DDSP violin model on EMF electrical noise
3. **§3.3 Loss Hacking** — Add an inharmonicity term to the objective; hear the violin drift toward metallic tones


---
## §3.1  Inspiring Set: Animal Vocalizations via a Limited Synthesizer

### Background — Hagiwara et al. (2022)

Hagiwara et al. synthesize animal vocalizations with a **differentiable modular synthesizer**.
The key idea: the synthesizer is *intentionally limited*:

> "Because the synthesizer cannot faithfully reproduce the target, gradient descent finds
> parameters that echo the target's most prominent acoustic feature while filling the rest
> with synthesizer artefacts."

This is the **inspiring-set** concept from Broad et al.: the creative output emerges from the
*gap* between what the model can produce and what the target demands.

### Our synth — `AnimalSynth`

A two-operator FM+AM synthesizer:

```
pitch trajectory (16 ctrl pts, E2–C6)
        ↓
FM carrier sin(∫2π f₀ dt + β·sin(2π·fₘ·t))
        ↓
× AM tremolo (1 + α·sin(2π·fₐₘ·t))   ← gives rhythmic chirp/croak patterns
        ↓
× amplitude envelope (16 ctrl pts)
        ↓
+ noise floor  →  audio
```

All parameters map from ℝ via tanh/sigmoid/exp — **no hard clamps that kill gradients**.
This means the optimizer can freely explore FM depth, AM rate, pitch trajectory.

**Deliberate limitations:**
- Max pitch ~1047 Hz (C6); bird calls at 2–4 kHz are reproduced an octave below — that gap *is* the inspiring-set effect
- 16 pitch control points cannot capture rapid trills
- Single FM operator cannot reproduce multi-formant frog calls or breath noise

Four animal categories: **cat, frog, chirping birds, crickets** — one clip each.


In [ ]:
ESC50_DIR = "/mnt/mariadata/datasets/ESC-50"

print("Loading animal vocalizations ...")
animals = load_esc50_animals(
    ESC50_DIR,
    categories={"cat", "frog", "chirping_birds", "crickets"},
    max_per_category=1,
    duration=2.0,
    sample_rate=SAMPLE_RATE,
)
print(f"Loaded {len(animals)} clips:")
for a in animals:
    print(f"  {a['category']:18s}  {a['filename']}")


### Optimization — 400 steps per clip (~3 s on CPU)

The optimizer uses cosine-annealed Adam on unconstrained parameters.
Watch the printed diagnostics: `f0` should move from the initialised ~295 Hz toward
the target's character; `fm` and `am` parameters adjust to match timbre and rhythm.

After fitting, compare spectrograms: the synth captures the *average* pitch and rhythmic
envelope but loses biological texture — that loss is the creative gain.


In [ ]:
fitted = []

for item in animals:
    cat = item['category']
    print(f"\n── Fitting: {cat} ({item['filename']}) ──")
    synth, losses = fit_synth_to_target(
        item['audio'],
        n_iter=400,
        lr=0.08,
        n_ctrl=16,
        sample_rate=SAMPLE_RATE,
        verbose=True,
    )
    with torch.no_grad():
        approx = synth()
    p = synth.get_param_dict()
    print(f"  → f0 {p['pitch_hz_range'][0]:.0f}–{p['pitch_hz_range'][1]:.0f} Hz  "
          f"fm {p['fm_depth']:.2f}@{p['fm_rate_hz']:.1f}Hz  "
          f"am {p['am_depth']:.2f}@{p['am_rate_hz']:.1f}Hz  "
          f"noise {p['noise_gain']:.3f}")
    fitted.append({
        'category': cat, 'filename': item['filename'],
        'target': item['audio'], 'approx': approx,
        'synth': synth, 'losses': losses, 'params': p,
    })

print("\nAll fits complete.")


In [ ]:
# ── Spectrogram comparison + loss curves ─────────────────────────────────────
n = len(fitted)
fig, axes = plt.subplots(n, 3, figsize=(15, 3.0 * n))
if n == 1: axes = axes[None, :]

for row, item in enumerate(fitted):
    ax_t, ax_a, ax_l = axes[row]
    specshow(item['target'], title=f"{item['category']} — TARGET",       ax=ax_t)
    specshow(item['approx'], title=f"{item['category']} — SYNTH APPROX", ax=ax_a)
    ax_l.plot(item['losses'], lw=1.5, color='steelblue')
    ax_l.set(xlabel='Iteration', ylabel='Spectral loss', title='Optimization loss')
    ax_l.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── Audio playback ────────────────────────────────────────────────────────────
for item in fitted:
    print(f"\n{'─'*55}")
    print(f"Category: {item['category']}  |  {item['filename']}")
    print("  Target:")
    play(item['target'])
    print("  Synth approximation:")
    play(item['approx'])


### Interactive parameter exploration

Sliders perturb the fitted parameters. Explore the creative neighbourhood around the
optimised solution: nearby configurations sound *related* to both the target and the fit,
but are neither.

- **Pitch shift** — transpose the entire pitch trajectory up/down
- **FM depth** — add/remove spectral richness (sidebands)
- **FM rate ×** — slow down / speed up FM modulation frequency
- **AM depth** — increase/decrease tremolo (chirp intensity)
- **AM rate ×** — slow / fast chirp repetition rate


In [ ]:
if not fitted:
    print("Run optimization cells above first.")
else:
    clip_opts = [(f"{r['category']} ({r['filename']})", i) for i, r in enumerate(fitted)]

    sel      = widgets.Dropdown(options=clip_opts, description='Clip:',
                                style={'description_width': '50px'})
    pitch_s  = widgets.FloatSlider(value=0.0,  min=-12.0, max=12.0, step=0.5,
                                    description='Pitch shift (st)',
                                    style={'description_width': '160px'})
    fmd_s    = widgets.FloatSlider(value=0.0,  min=-4.0,  max=4.0,  step=0.1,
                                    description='FM depth add',
                                    style={'description_width': '160px'})
    fmr_s    = widgets.FloatSlider(value=1.0,  min=0.2,   max=5.0,  step=0.05,
                                    description='FM rate ×',
                                    style={'description_width': '160px'})
    amd_s    = widgets.FloatSlider(value=0.0,  min=-0.8,  max=0.8,  step=0.05,
                                    description='AM depth add',
                                    style={'description_width': '160px'})
    amr_s    = widgets.FloatSlider(value=1.0,  min=0.2,   max=5.0,  step=0.05,
                                    description='AM rate ×',
                                    style={'description_width': '160px'})
    btn      = widgets.Button(description='▶  Synthesise & Play',
                               button_style='primary',
                               layout=widgets.Layout(width='200px'))
    out_w    = widgets.Output()

    def on_synth(b):
        with out_w:
            out_w.clear_output(wait=True)
            idx = sel.value
            base = fitted[idx]['synth']
            p = base.perturb(
                pitch_semitones=pitch_s.value,
                fm_depth_add=fmd_s.value,
                fm_rate_mult=fmr_s.value,
                am_depth_add=amd_s.value,
                am_rate_mult=amr_s.value,
            )
            with torch.no_grad():
                audio = p()
            fig, axes = plt.subplots(1, 2, figsize=(12, 2.5))
            specshow(fitted[idx]['approx'], title='Fitted (reference)', ax=axes[0])
            specshow(audio,                 title='Perturbed',          ax=axes[1])
            plt.tight_layout()
            plt.show()
            play(audio)

    btn.on_click(on_synth)
    display(widgets.VBox([sel, pitch_s, fmd_s, fmr_s, amd_s, amr_s, btn, out_w]))


---
## §3.2  Divergent Fine-tuning: Violin → EMF Noise

### Concept: "Find a Fertile Maladaptation"

Fine-tuning on foreign data does not simply teach new content — it *distorts* existing
representations. Stopped **before convergence**, the model lives in a liminal zone:

- It still carries the violin's harmonic vocabulary
- But it is being pulled toward the timbre of the new source
- That in-between zone — under-adapted, partially deformed — is where the most interesting sounds live

This is the training-time version of the "interesting region" from Broad et al.: the creative
potential sits at the **edge of maladaptation**, not at convergence.

### Data: EMF electrical noise

We fine-tune on **electromagnetic interference** recordings — 50 Hz hum and RF noise.
Maximally different from violin: no pitch, complex noise structure, harmonic grid at 50/100/… Hz.

Because EMF has no reliable pitch (CREPE periodicity ≈ 0.001), we assign **fixed f0 = 100 Hz**
to all fine-tuning chunks.  The model learns "at 100 Hz, produce what the training data sounds like"
— and that timbre is electrical buzz.

### Probe — real violin features

The probe uses **actual f0 + loudness curves extracted from a violin clip** (pre-computed in
`samples/processed/`).  This means:
- At step 0 (baseline) the probe sounds like a recognisable, if imperfect, violin phrase
- As fine-tuning progresses, the same pitch/loudness trajectory sounds increasingly like noise-violin hybrid

This makes divergence immediately audible — not just a change in tone colour of a sine wave.


In [ ]:
# ── Set the fine-tuning audio directory ──────────────────────────────────────
FINETUNE_DIR_DEFAULT = "/mnt/mariadata/datasets/emf-noises-freesound"
SOURCE_TAG = "violin"
TARGET_TAG = "emf"

dir_widget = widgets.Text(
    value=FINETUNE_DIR_DEFAULT,
    description='Fine-tune dir:',
    layout=widgets.Layout(width='700px'),
    style={'description_width': '120px'},
)
display(dir_widget)
print("Edit the path above if needed, then run the next cell.")


In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
FINETUNE_DIR   = dir_widget.value
CHUNK_DURATION = 2.0      # seconds per training clip
FIXED_F0_HZ    = 100.0   # constant f0 — EMF is unpitched
MAX_CHUNKS     = 20
BATCH_SIZE     = 2
N_EPOCHS       = 5
LR             = 3e-4
SAVE_EVERY     = 10
FFT_SIZES_FT   = (2048, 1024, 512, 256)

# ── Probe: real violin features from a processed clip ─────────────────────────
PROBE_CLIP_IDX = 5  # pick a clip with varied pitch — change to try others

probe_f0, probe_loud, probe_ref_audio = make_probe_from_clip(
    PROCESSED_DIR, clip_idx=PROBE_CLIP_IDX)

print(f"Probe: f0 {probe_f0.min():.0f}–{probe_f0.max():.0f} Hz, "
      f"loudness {probe_loud.min():.1f}–{probe_loud.max():.1f} dB, "
      f"duration {probe_ref_audio.shape[0]/SAMPLE_RATE:.1f}s")

print("\nOriginal violin audio (reference):")
play(probe_ref_audio)

# ── Load chunks ───────────────────────────────────────────────────────────────
print(f"\nLoading EMF chunks from: {FINETUNE_DIR}")
chunks = load_audio_chunks(
    FINETUNE_DIR,
    chunk_duration=CHUNK_DURATION,
    sample_rate=SAMPLE_RATE,
    max_chunks=MAX_CHUNKS,
    fixed_f0_hz=FIXED_F0_HZ,
    hop_length=HOP_LENGTH,
)
print(f"Loaded {len(chunks)} chunks × {CHUNK_DURATION}s")

# ── Model + optimiser ─────────────────────────────────────────────────────────
ft_model, ft_config = load_baseline(freeze=False)
ft_model.train()
ft_optimizer = torch.optim.Adam(ft_model.parameters(), lr=LR)
ft_criterion = MultiScaleSpectralLoss(fft_sizes=FFT_SIZES_FT)

print(f"Model params: {sum(p.numel() for p in ft_model.parameters()):,}")
print("Ready.  Original violin probe above — compare it to the probes below after training.")


### Fine-tuning loop

After every `SAVE_EVERY` steps: save a checkpoint and render the probe.
The probe uses the real violin f0/loudness, so you hear the exact same musical gesture
played through progressively more divergent model weights.


In [ ]:
ft_probe_audios = []
ft_loss_history = []
ft_step = 0


def _sample_batch(chunk_list, bs):
    idxs = torch.randperm(len(chunk_list))[:bs].tolist()
    n_min = min(chunk_list[i]['f0_hz'].shape[0] for i in idxs)
    return {
        'audio':    torch.stack([chunk_list[i]['audio']              for i in idxs]),
        'f0_hz':    torch.stack([chunk_list[i]['f0_hz'][:n_min]     for i in idxs]),
        'loudness': torch.stack([chunk_list[i]['loudness'][:n_min]  for i in idxs]),
    }


@torch.no_grad()
def render_probe(model):
    model.eval()
    # Use only first 2s of probe for speed
    n2s = 2 * SAMPLE_RATE // HOP_LENGTH
    f0   = probe_f0[:, :n2s]
    loud = probe_loud[:, :n2s]
    audio = model(f0, loud)['audio'].squeeze()
    model.train()
    return audio


# ── Baseline probe ────────────────────────────────────────────────────────────
ft_probe_audios.append(('step 0 (baseline)', render_probe(ft_model)))
print(f"Baseline probe rendered.")
print(f"Starting: {N_EPOCHS} epochs × {len(chunks) // BATCH_SIZE} steps\n")

t_start = time.time()

for epoch in range(N_EPOCHS):
    steps_this_epoch = len(chunks) // BATCH_SIZE
    epoch_loss = 0.0
    for _ in range(steps_this_epoch):
        batch = _sample_batch(chunks, BATCH_SIZE)
        lv = finetune_step(ft_model, ft_optimizer, ft_criterion, batch)
        ft_loss_history.append((ft_step, lv))
        epoch_loss += lv
        ft_step += 1
        if ft_step % SAVE_EVERY == 0:
            pa = render_probe(ft_model)
            ft_probe_audios.append((f'step {ft_step}', pa))
            ckpt = save_divergent_checkpoint(
                ft_model, ft_step, SOURCE_TAG, TARGET_TAG, MODELS_DIR, ft_config)
            print(f"  ✓ {os.path.basename(ckpt)}")
    print(f"Epoch {epoch+1}/{N_EPOCHS}  avg={epoch_loss/steps_this_epoch:.4f}  "
          f"elapsed={time.time()-t_start:.0f}s")

print(f"\nDone. Steps: {ft_step}")


In [ ]:
# ── Loss curve ────────────────────────────────────────────────────────────────
if ft_loss_history:
    steps, vals = zip(*ft_loss_history)
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(steps, vals, lw=1.2, color='crimson', alpha=0.8)
    ax.set(xlabel='Step', ylabel='Spectral loss', title='Fine-tuning loss — violin → EMF')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Probe spectrogram strip ───────────────────────────────────────────────────
n_p = len(ft_probe_audios)
if n_p:
    fig, axes = plt.subplots(1, n_p, figsize=(4 * n_p, 3), sharey=True)
    if n_p == 1: axes = [axes]
    for ax, (label, audio) in zip(axes, ft_probe_audios):
        specshow(audio, title=label, ax=ax)
    plt.suptitle('Probe output — same real violin f0/loudness through changing weights', y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Listen to divergence accumulating ────────────────────────────────────────
print("Original violin audio (for comparison):")
play(probe_ref_audio[:2*SAMPLE_RATE])

for label, audio in ft_probe_audios:
    print(f"  {label}:")
    play(audio)


---
## §3.3  Loss Hacking: Inharmonicity Penalty

### Concept

Loss modification is the most direct form of training-time active divergence: we **rewrite the
objective function** itself.  Same data, same architecture — but the thing the model is
optimised *for* changes.

We add an **inharmonicity term** to the multi-scale spectral loss.  Rather than always rewarding
faithful reconstruction, we simultaneously reward *spreading harmonic energy across more overtones*.

### Mathematical definition

The DDSP decoder outputs `harmonic_dist` — a softmax distribution
$\mathbf{p} \in \Delta^{K-1}$ over $K = 100$ harmonics.

A **concentrated** distribution (energy mainly in harmonics 1–3) = pure violin.
A **spread** distribution (energy across all 100 harmonics) = metallic, bell-like, dense.

We maximise the **Shannon entropy**:

$$H(\mathbf{p}) = -\sum_{k=1}^{K} p_k \log(p_k + \varepsilon)$$

The loss term is **negative entropy** (minimising = maximising spread):

$$\mathcal{L}_{\text{inh}}(\mathbf{p}) = -H(\mathbf{p})$$

Combined objective:  $\mathcal{L} = \mathcal{L}_{\text{spectral}} + \lambda\,\mathcal{L}_{\text{inh}}$

### Probe — same real violin features as §3.2

Using the same real (f0, loudness) probe makes the timbre drift directly audible against
the reference violin performance.


In [ ]:
inh_weight_widget = widgets.FloatSlider(
    value=5.0, min=0.0, max=30.0, step=0.5,
    description='λ (inharmonicity weight):',
    style={'description_width': '220px'},
    layout=widgets.Layout(width='600px'),
)
display(inh_weight_widget)
print("Adjust λ, then run the training loop cell.")
print("  λ = 0 :  pure reconstruction (baseline)")
print("  λ = 5 :  mild inharmonicity — slightly metallic")
print("  λ ≥15 :  strong spread — bell/buzz-like")


In [ ]:
INH_WEIGHT          = inh_weight_widget.value
N_EPOCHS_INH        = 5
STEPS_PER_EPOCH_INH = 30
BATCH_SIZE_INH      = 2
LR_INH              = 1e-4
SAVE_EVERY_INH      = 10
FFT_SIZES_INH       = (2048, 1024, 512, 256)

print(f"λ = {INH_WEIGHT}")

# ── Fresh model from baseline ─────────────────────────────────────────────────
inh_model, inh_config = load_baseline(freeze=False)
inh_model.train()
inh_optimizer = torch.optim.Adam(inh_model.parameters(), lr=LR_INH)
inh_spec_crit = MultiScaleSpectralLoss(fft_sizes=FFT_SIZES_INH)
inh_loss_fn   = InharmonicityLoss()

# ── Same violin dataset as baseline training ──────────────────────────────────
violin_ds = URMPViolinDataset(PROCESSED_DIR)
violin_loader = DataLoader(violin_ds, batch_size=BATCH_SIZE_INH, shuffle=True,
                            collate_fn=URMPViolinDataset.collate_fn, drop_last=True)

# ── Probe — same real violin clip as §3.2 ────────────────────────────────────
# probe_f0, probe_loud, probe_ref_audio are already loaded above
print(f"Using probe clip {PROBE_CLIP_IDX}: f0 {probe_f0.min():.0f}–{probe_f0.max():.0f} Hz")
print(f"Dataset: {len(violin_ds)} clips | Params: {sum(p.numel() for p in inh_model.parameters()):,}")


### Training loop with inharmonicity loss

Two loss scalars tracked at every step:
- `spectral` = reconstruction fidelity (should **rise** as inharmonicity pulls the model away)
- `inh` = −entropy of harmonic_dist (should **fall** as harmonics spread)


In [ ]:
inh_spectral_hist = []
inh_inh_hist      = []
inh_probe_audios  = []
inh_step = 0
data_iter = iter(violin_loader)


@torch.no_grad()
def render_inh_probe(model):
    model.eval()
    n2s = 2 * SAMPLE_RATE // HOP_LENGTH
    f0   = probe_f0[:, :n2s]
    loud = probe_loud[:, :n2s]
    audio = model(f0, loud)['audio'].squeeze()
    model.train()
    return audio


# ── Baseline probe ────────────────────────────────────────────────────────────
inh_probe_audios.append(('λ=0 (baseline)', render_inh_probe(inh_model)))
print(f"Starting §3.3: λ={INH_WEIGHT}, {N_EPOCHS_INH} epochs\n")

t_start = time.time()

for epoch in range(N_EPOCHS_INH):
    for _ in range(STEPS_PER_EPOCH_INH):
        try:
            batch = next(data_iter)
        except StopIteration:
            data_iter = iter(violin_loader)
            batch = next(data_iter)

        inh_model.train()
        out    = inh_model(batch['f0_hz'], batch['loudness'])
        s_loss = inh_spec_crit(out['audio'], batch['audio'])
        i_loss = inh_loss_fn(out['harmonic_dist'])
        total  = s_loss + INH_WEIGHT * i_loss

        inh_optimizer.zero_grad()
        total.backward()
        torch.nn.utils.clip_grad_norm_(inh_model.parameters(), 1.0)
        inh_optimizer.step()

        inh_spectral_hist.append((inh_step, s_loss.item()))
        inh_inh_hist.append((inh_step, i_loss.item()))
        inh_step += 1

        if inh_step % SAVE_EVERY_INH == 0:
            pa = render_inh_probe(inh_model)
            inh_probe_audios.append((f'step {inh_step}', pa))
            ckpt = save_inharmonic_checkpoint(inh_model, inh_step, INH_WEIGHT,
                                               MODELS_DIR, inh_config)
            print(f"  ✓ {os.path.basename(ckpt)}")

    avg_s = np.mean([v for _, v in inh_spectral_hist[-STEPS_PER_EPOCH_INH:]])
    avg_i = np.mean([v for _, v in inh_inh_hist[-STEPS_PER_EPOCH_INH:]])
    print(f"Epoch {epoch+1}/{N_EPOCHS_INH}  spectral={avg_s:.4f}  inh={avg_i:.4f}  "
          f"elapsed={time.time()-t_start:.0f}s")

print(f"\nDone. Steps: {inh_step}")


In [ ]:
# ── Dual loss curves ──────────────────────────────────────────────────────────
if inh_spectral_hist and inh_inh_hist:
    ss, vs = zip(*inh_spectral_hist)
    si, vi = zip(*inh_inh_hist)
    fig, ax1 = plt.subplots(figsize=(11, 3.5))
    ax2 = ax1.twinx()
    l1, = ax1.plot(ss, vs, color='tomato',    lw=1.5, label='Spectral reconstruction loss ↑')
    l2, = ax2.plot(si, vi, color='steelblue', lw=1.5, label='Inharmonicity loss (−H) ↓', alpha=0.85)
    ax1.set(xlabel='Step', ylabel='Spectral loss',
            title=f'Loss hacking λ={INH_WEIGHT}: reconstruction ↑ vs inharmonicity ↓')
    ax2.set_ylabel('Inharmonicity loss (negative entropy)')
    ax1.grid(True, alpha=0.3)
    ax1.legend(handles=[l1, l2], loc='upper right')
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Probe spectrogram strip ───────────────────────────────────────────────────
n_p = len(inh_probe_audios)
if n_p:
    fig, axes = plt.subplots(1, n_p, figsize=(4 * n_p, 3), sharey=True)
    if n_p == 1: axes = [axes]
    for ax, (label, audio) in zip(axes, inh_probe_audios):
        specshow(audio, title=label, ax=ax)
    plt.suptitle(f'Inharmonicity probe (λ={INH_WEIGHT}) — same violin f0/loudness', y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Listen to inharmonicity accumulating ─────────────────────────────────────
print("Original violin audio (reference):")
play(probe_ref_audio[:2*SAMPLE_RATE])

for label, audio in inh_probe_audios:
    print(f"  {label}:")
    play(audio)


---
## Summary

| Technique | What changes | Timbre trajectory |
|-----------|-------------|-------------------|
| **§3.1 Inspiring set** | Synth params per sample | Biological texture replaced by FM+AM artefact; rhythmic character echoed |
| **§3.2 Divergent fine-tuning** | Decoder weights → EMF domain | Same violin melody sounds increasingly like noise-buzz hybrid |
| **§3.3 Loss hacking** | Training objective → inharmonicity | Same violin melody drifts toward metallic/bell-like as reconstruction degrades |

**Common thread: controlled maladaptation.** The creative potential sits at the *edge of arrival*, not at convergence.
